<a href="https://colab.research.google.com/github/loudamn78-gif/thakaa-flow-training/blob/main/week2/day11/day11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## Task 1
## 1. بطاقة الداتا
| العمود | معناه | النوع | Target أم Feature |
|---|---|---|---|
| `Loan_ID` | رقم تعريفي لطلب القرض | معرّف | Feature (لا يفيد التنبؤ) |
| `Gender` | جنس المتقدم | تصنيفي | Feature |
| `Married` | هل المتقدم متزوج | تصنيفي | Feature |
| `Dependents` | عدد المعالين | تصنيفي | Feature |
| `Education` | المستوى التعليمي (خريج أو غير خريج) | تصنيفي | Feature |
| `Self_Employed` | هل يعمل لحسابه الخاص | تصنيفي | Feature |
| `ApplicantIncome` | دخل المتقدم | رقمي | Feature |
| `CoapplicantIncome` | دخل المتقدم المشارك | رقمي | Feature |
| `LoanAmount` | مبلغ القرض (بالآلاف) | رقمي | Feature |
| `Loan_Amount_Term` | مدة القرض (بالأشهر) | رقمي | Feature |
| `Credit_History` | هل لديه سجل ائتماني جيد (1 أو 0) | تصنيفي | Feature |
| `Property_Area` | منطقة العقار (ريفي، شبه حضري، حضري) | تصنيفي | Feature |
| `Loan_Status` | هل وُوفق على القرض (Y أو N) | تصنيفي | **Target** |


2. فرضيات قبل الأرقام
- Credit_History: أتوقع تأثرها بقوة على الموافقة، لأن اللي عنده سجل ائتماني كويس غالباً يكون ملتزم بالسداد.
- ApplicantIncome: تؤثر بقوة، وكل ما زاد دخل الشخص معناها قدرته على السداد بتكون أعلى.
- CoapplicantIncome: تأثيرها متوسط، لأن الدخل الإضافي يعتبر دعم كويس ويزيد نسبة قبول القرض.
- LoanAmount: تأثيرها عكسي وقوي، المبالغ الضخمة ترفع نسبة الخطر وتصعب الموافقة.
- Education: تأثيرها متوسط، لأن التعليم العالي غالباً يربطونه بالاستقرار الوظيفي والدخل المرتفع.
-Property_Area: تأثيرها متوسط، لأن طبيعة الأصول والمخاطر تختلف بين المناطق الحضرية والريفية.
- Dependents: تأثيرها عكسي ومتوسط، زيادة عدد المعالين تعني مصاريف أكثر على العميل.
- Married / Gender / Self_Employed: أحس تأثيرها بسيط مقارنة بالدخل والسجل المالي المباشر.

3. عمود LoanAmount و Loan_Amount_Term
- مرة مهم نعرف وحدة القياس قبل ما نبدأ نحلل عشان نفهم الأرقام صح؛ مثلاً LoanAmount بالآلاف و Loan_Amount_Term بالأشهر. لو ما انتبهنا للنقطة هذه بنلخبط ونقارن أرقام غلط، زي إننا نحسب مبلغ 146 ريال وهو بالحقيقة 146,000 ريال.

4. سؤال أخلاقي: استخدام عمود Gender
أشوف إنه مو عادل أبداً نستخدم عمود الجنس (Gender) في موديل القروض. المفروض التقييم يعتمد بس على الجدارة المالية للعميل مثل دخله وسجله الائتماني وقيمة القرض مو على جنسه. لأن إدخال هذا العمود راح يسبب انحياز (Bias) في الموديل ويميز بين المتقدمين بطريقة تخالف العدالة والمنطق الأخلاقي.

## Task 2


In [37]:
import pandas as pd

df = pd.read_csv("loan.csv")

df.info()
print(df.shape)
print(df.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB
(614, 13)
    Loan_ID Gender Married Dependents Education Self_Employed  \
0  LP001002   Male      No          0

In [38]:
# 1. info و shape و head
df.info()
print(df.shape)
print(df.head(3))

# 2. القيم الناقصة: العدد والنسبة، مرتبة تنازليًا
missing = pd.DataFrame({
    "count": df.isna().sum(),
    "percent": (df.isna().mean() * 100).round(2)
})
print(missing[missing["count"] > 0].sort_values("count", ascending=False))

# 3. الصفوف المكررة وعدد القيم المختلفة
print("الصفوف المكررة:", df.duplicated().sum())
print(df.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB
(614, 13)
    Loan_ID Gender Married Dependents Education Self_Employed  \
0  LP001002   Male      No          0

In [39]:
# 4. describe للأعمدة الرقمية
print(df.describe())

# 5. value_counts لكل عمود تصنيفي
for col in ["Gender", "Married", "Dependents", "Education",
            "Self_Employed", "Credit_History", "Property_Area", "Loan_Status"]:
    print(df[col].value_counts(dropna=False))
    print()

# 6. نسبة الطلبات المقبولة والمرفوضة
print(df["Loan_Status"].value_counts(normalize=True).round(3))

       ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
count       614.000000         614.000000  592.000000         600.00000   
mean       5403.459283        1621.245798  146.412162         342.00000   
std        6109.041673        2926.248369   85.587325          65.12041   
min         150.000000           0.000000    9.000000          12.00000   
25%        2877.500000           0.000000  100.000000         360.00000   
50%        3812.500000        1188.500000  128.000000         360.00000   
75%        5795.000000        2297.250000  168.000000         360.00000   
max       81000.000000       41667.000000  700.000000         480.00000   

       Credit_History  
count      564.000000  
mean         0.842199  
std          0.364878  
min          0.000000  
25%          1.000000  
50%          1.000000  
75%          1.000000  
max          1.000000  
Gender
Male      489
Female    112
NaN        13
Name: count, dtype: int64

Married
Yes    398
No     213
Na

In [40]:
print(df[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]].describe().round(2))

       ApplicantIncome  CoapplicantIncome  LoanAmount
count           614.00             614.00      592.00
mean           5403.46            1621.25      146.41
std            6109.04            2926.25       85.59
min             150.00               0.00        9.00
25%            2877.50               0.00      100.00
50%            3812.50            1188.50      128.00
75%            5795.00            2297.25      168.00
max           81000.00           41667.00      700.00


## 4. ملاحظات describe
في الأعمدة الرقمية المتوسط أعلى من الوسيط:
- `ApplicantIncome`: المتوسط 5403.46 والوسيط 3812.50، والقيمة القصوى 81000.
- `CoapplicantIncome`: المتوسط 1621.25 والوسيط 1188.50، والقيمة القصوى 41667.
- `LoanAmount`: المتوسط 146.41 والوسيط 128، والقيمة القصوى 700.

هذا يدل على وجود قيم عالية جدًا (Outliers) تسحب المتوسط للأعلى، فالوسيط أصدق تمثيلًا للقيمة المعتادة.

### 5. ملاحظات value_counts
- `Dependents` فيه القيمة `3+` وهي نص وليست رقمًا.
- `Self_Employed` غير متوازن: 500 No مقابل 82 Yes.
- `Credit_History` رقمي لكنه تصنيفي في معناه، وفيه 50 قيمة ناقصة.

### 6. نسبة الموافقة
الطلبات المقبولة 68.7% (422) والمرفوضة 31.3% (192).

## Task 3


In [41]:
# 3. LoanAmount: الوسيط
print("المتوسط:", df["LoanAmount"].mean().round(2), "| الوسيط:", df["LoanAmount"].median())
print("الناقص قبل:", df["LoanAmount"].isna().sum())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

# 4. Loan_Amount_Term: الأكثر تكرارًا
print(df["Loan_Amount_Term"].value_counts(dropna=False).head())
print("الأكثر تكرارًا:", df["Loan_Amount_Term"].mode()[0], "(بالأشهر = 30 سنة لو 360)")
df["Loan_Amount_Term"] = df["Loan_Amount_Term"].fillna(df["Loan_Amount_Term"].mode()[0])

# 5. Credit_History: نسبة الموافقة للصفوف الناقصة قبل أي قرار
missing_ch = df[df["Credit_History"].isna()]
print("نسبة الموافقة للصفوف الناقصة:", (missing_ch["Loan_Status"] == "Y").mean().round(3))
print("نسبة الموافقة للصفوف 0:", (df[df["Credit_History"] == 0]["Loan_Status"] == "Y").mean().round(3))
print("نسبة الموافقة للصفوف 1:", (df[df["Credit_History"] == 1]["Loan_Status"] == "Y").mean().round(3))

المتوسط: 146.41 | الوسيط: 128.0
الناقص قبل: 22
Loan_Amount_Term
360.0    512
180.0     44
480.0     15
NaN       14
300.0     13
Name: count, dtype: int64
الأكثر تكرارًا: 360.0 (بالأشهر = 30 سنة لو 360)
نسبة الموافقة للصفوف الناقصة: 0.74
نسبة الموافقة للصفوف 0: 0.079
نسبة الموافقة للصفوف 1: 0.796


In [42]:
# تجربة مؤقتة: -1 تعني "ناقص" للمقارنة
temp = df["Credit_History"].fillna(-1)
print(df.groupby(temp)["Loan_Status"].apply(lambda s: (s == "Y").mean()).round(3))

# القرار النهائي: mode (قيمة 1) لأن نسبة موافقة الناقص قريبة من مجموعة 1
print("الناقص قبل:", df["Credit_History"].isna().sum())
df["Credit_History"] = df["Credit_History"].fillna(df["Credit_History"].mode()[0])
print("الناقص بعد:", df["Credit_History"].isna().sum())

Credit_History
-1.0    0.740
 0.0    0.079
 1.0    0.796
Name: Loan_Status, dtype: float64
الناقص قبل: 50
الناقص بعد: 0


In [43]:
# 6. Outliers بطريقة IQR
for col in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < low) | (df[col] > high)).sum()
    print(col, "| الحد الأعلى:", round(high, 2), "| عدد الـ Outliers:", n)

# 7. تحويل Loan_Status إلى أرقام
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})
print(df["Loan_Status"].value_counts())

ApplicantIncome | الحد الأعلى: 10171.25 | عدد الـ Outliers: 50
CoapplicantIncome | الحد الأعلى: 5743.12 | عدد الـ Outliers: 18
LoanAmount | الحد الأعلى: 261.5 | عدد الـ Outliers: 41
Loan_Status
1    422
0    192
Name: count, dtype: int64


In [44]:
# 8. التأكد من عدم وجود نقص، ثم الحفظ
print(df.isna().sum())
df.to_csv("loan_clean.csv", index=False)

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            0
Loan_Amount_Term      0
Credit_History        0
Property_Area         0
Loan_Status           0
dtype: int64


In [45]:
# 1. الأعمدة التصنيفية: mode
for col in ["Gender", "Married", "Self_Employed"]:
    print(col, "| الناقص:", df[col].isna().sum(), "| الأكثر تكرارًا:", df[col].mode()[0])
    df[col] = df[col].fillna(df[col].mode()[0])

# 2. Dependents: املأ النقص ثم حوّل 3+ إلى 3 واجعل العمود رقمًا
print("Dependents قبل:", df["Dependents"].value_counts(dropna=False).to_dict())
df["Dependents"] = df["Dependents"].fillna(df["Dependents"].mode()[0])
df["Dependents"] = df["Dependents"].replace("3+", "3").astype(int)
print("Dependents بعد:", df["Dependents"].value_counts().to_dict())

# 8. التأكد ثم الحفظ مرة ثانية
print(df.isna().sum())
df.to_csv("loan_clean.csv", index=False)

Gender | الناقص: 13 | الأكثر تكرارًا: Male
Married | الناقص: 3 | الأكثر تكرارًا: Yes
Self_Employed | الناقص: 32 | الأكثر تكرارًا: No
Dependents قبل: {'0': 345, '1': 102, '2': 101, '3+': 51, nan: 15}
Dependents بعد: {0: 360, 1: 102, 2: 101, 3: 51}
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


### 9. Decision Log

| # | العمود / المشكلة | القرار | السبب (بالأرقام) | الصفوف المتأثرة |
|---|---|---|---|---|
| 1 | `Gender` ناقص | `mode` (Male) | قيمة تصنيفية، والأكثر تكرارًا Male (489) | 13 |
| 2 | `Married` ناقص | `mode` (Yes) | قيمة تصنيفية، Yes (398) مقابل No (213) | 3 |
| 3 | `Self_Employed` ناقص | `mode` (No) | No (500) مقابل Yes (82) | 32 |
| 4 | `Dependents` ناقص و`3+` نص | `mode` (0) ثم تحويل `3+` إلى 3 و`astype(int)` | عدد صحيح بطبيعته، و0 الأكثر تكرارًا (345) | 15 ناقصة + 51 محوّلة |
| 5 | `LoanAmount` ناقص | الوسيط (128) | المتوسط 146.41 مسحوب بقيم عالية (الأعلى 700)، والعمود مستمر | 22 |
| 6 | `Loan_Amount_Term` ناقص | `mode` (360 شهر) | 360 هو الأكثر تكرارًا (512 من 614)، والمدة عدد محدود من القيم | 14 |
| 7 | `Credit_History` ناقص | `mode` (1) | نسبة موافقة الناقص 0.74 قريبة من مجموعة 1 (0.796) وبعيدة عن مجموعة 0 (0.079)، فالتعبئة بـ 0 كانت ستكون خطأ | 50 |
| 8 | Outliers في `ApplicantIncome` و`CoapplicantIncome` و`LoanAmount` | أبقيتها | حالات حقيقية (دخل أو قرض عالٍ)، والداتا صغيرة (614) | 50 و18 و41 |
| 9 | `Loan_Status` نص (Y/N) | `map` إلى 1 و0 | يسهّل التحليل والموديل | 614 (422 موافقة و192 رفض) |

**ملاحظات:**
- ما حذفت أي صف ولا أي عمود، وبقي الشكل (614, 13).
- الـ Outliers بطريقة IQR: الحد الأعلى للدخل 10171.25، وللدخل المشترك 5743.12، ولمبلغ القرض 261.5.
- بعد كل القرارات: نقص = 0 في كل الأعمدة، وحُفظ الملف في `loan_clean.csv`.

## Task 4


In [46]:

df_raw = pd.read_csv("loan.csv")

# 1. قبل وبعد: الشكل، النقص، ونسبة الموافقة
print("الشكل قبل:", df_raw.shape, "| بعد:", df.shape)
print("إجمالي النقص قبل:", df_raw.isna().sum().sum(), "| بعد:", df.isna().sum().sum())
print("نسبة الموافقة قبل:", (df_raw["Loan_Status"] == "Y").mean().round(3),
      "| بعد:", df["Loan_Status"].mean().round(3))

# 2. LoanAmount: المتوسط والوسيط قبل وبعد
print("LoanAmount قبل | المتوسط:", df_raw["LoanAmount"].mean().round(2), "| الوسيط:", df_raw["LoanAmount"].median())
print("LoanAmount بعد | المتوسط:", df["LoanAmount"].mean().round(2), "| الوسيط:", df["LoanAmount"].median())

# 3. Credit_History: التوزيع قبل وبعد
print(df_raw["Credit_History"].value_counts(normalize=True, dropna=False).round(3))
print(df["Credit_History"].value_counts(normalize=True).round(3))

الشكل قبل: (614, 13) | بعد: (614, 13)
إجمالي النقص قبل: 149 | بعد: 0
نسبة الموافقة قبل: 0.687 | بعد: 0.687
LoanAmount قبل | المتوسط: 146.41 | الوسيط: 128.0
LoanAmount بعد | المتوسط: 145.75 | الوسيط: 128.0
Credit_History
1.0    0.774
0.0    0.145
NaN    0.081
Name: proportion, dtype: float64
Credit_History
1.0    0.855
0.0    0.145
Name: proportion, dtype: float64


## 4. رسالة لمدير المخاطر
1. الداتا فيها 614 طلب قرض، وكان فيها 149 قيمة ناقصة موزعة على 7 أعمدة.
2. عبّأت الناقص بقيم مناسبة لكل عمود (الأكثر تكرارًا للتصنيفي، والوسيط للمبالغ)، وما حذفت أي طلب.
3. النتيجة ما تغيّرت: نسبة الموافقة بقيت 68.7%، ووسيط مبلغ القرض بقي 128.
4. الداتا الآن جاهزة للتحليل، وحُفظت في `loan_clean.csv`.
5. الخطر الوحيد: 50 طلبًا بلا سجل ائتماني معروف عبّأتها بـ 1 (لأن نسبة موافقتها 74% تشبه من عندهم سجل جيد)، فلو كان هذا الافتراض خاطئًا تتأثر الدقة، والأفضل التأكد منه من مصدر الداتا.

##Task 5


In [47]:
fossils = {
    "fossil_id": ["F01", "F02", "F03", "F04", "F05", "F02", "F06", "F07", "F08", "F09"],
    "species": ["T-Rex", "Triceratops", "t-rex", "Velociraptor", "Stegosaurus", "Triceratops",
                "Brachiosaurus", "T-Rex", "Velociraptor", "Stegosaurus"],
    "age_million_years": [68, 67, 66, 75, -150, 67, 150, 68, None, 155],
    "length": ["12 m", "9 m", "40 ft", "2 m", "9 m", "9 m", "26 m", "12.3 m", "6.6 ft", "9 m"],
    "found_in": ["USA", "USA", "Canada", "Mongolia", "USA", "USA", "Tanzania", "USA", "Mongolia", "Portugal"],
}
fos = pd.DataFrame(fossils)

# 1. المكرر وتوحيد أسماء الأنواع
print("المكرر:", fos.duplicated().sum())
print(fos["species"].value_counts())
fos = fos.drop_duplicates()
fos["species"] = fos["species"].str.strip().str.title()
print(fos["species"].value_counts())
print(fos.shape)

المكرر: 1
species
T-Rex            2
Triceratops      2
Velociraptor     2
Stegosaurus      2
t-rex            1
Brachiosaurus    1
Name: count, dtype: int64
species
T-Rex            3
Velociraptor     2
Stegosaurus      2
Triceratops      1
Brachiosaurus    1
Name: count, dtype: int64
(9, 5)


In [48]:
# 2. العمر السالب: ما سببه؟
print(fos[fos["age_million_years"] < 0])
fos.loc[fos["age_million_years"] < 0, "age_million_years"] = fos["age_million_years"].abs()

# 3. العمر الناقص: وسيط نفس النوع (وليس وسيط كل الحفريات)
print("الناقص قبل:", fos["age_million_years"].isna().sum())
fos["age_million_years"] = fos["age_million_years"].fillna(
    fos.groupby("species")["age_million_years"].transform("median"))
print("الناقص بعد:", fos["age_million_years"].isna().sum())
print(fos[["fossil_id", "species", "age_million_years"]])

  fossil_id      species  age_million_years length found_in
4       F05  Stegosaurus             -150.0    9 m      USA
الناقص قبل: 1
الناقص بعد: 0
  fossil_id        species  age_million_years
0       F01          T-Rex               68.0
1       F02    Triceratops               67.0
2       F03          T-Rex               66.0
3       F04   Velociraptor               75.0
4       F05    Stegosaurus              150.0
6       F06  Brachiosaurus              150.0
7       F07          T-Rex               68.0
8       F08   Velociraptor               75.0
9       F09    Stegosaurus              155.0


In [49]:
# 4. فصل الرقم عن الوحدة وتحويل الكل إلى متر
fos[["num", "unit"]] = fos["length"].str.split(" ", expand=True)
fos["num"] = fos["num"].astype(float)
print(fos[["length", "num", "unit"]])
fos["length_m"] = fos.apply(
    lambda r: r["num"] * 0.3048 if r["unit"] == "ft" else r["num"], axis=1).round(2)
print(fos[["length", "length_m"]])

# 5. متوسط الطول لكل نوع، وأقدم نوع، والدولة الأكثر عددًا
print(fos.groupby("species")["length_m"].mean().round(2).sort_values(ascending=False))
oldest = fos.loc[fos["age_million_years"].idxmax(), ["species", "age_million_years"]]
print("أقدم نوع:", oldest.to_dict())
print(fos["found_in"].value_counts())

   length   num unit
0    12 m  12.0    m
1     9 m   9.0    m
2   40 ft  40.0   ft
3     2 m   2.0    m
4     9 m   9.0    m
6    26 m  26.0    m
7  12.3 m  12.3    m
8  6.6 ft   6.6   ft
9     9 m   9.0    m
   length  length_m
0    12 m     12.00
1     9 m      9.00
2   40 ft     12.19
3     2 m      2.00
4     9 m      9.00
6    26 m     26.00
7  12.3 m     12.30
8  6.6 ft      2.01
9     9 m      9.00
species
Brachiosaurus    26.00
T-Rex            12.16
Stegosaurus       9.00
Triceratops       9.00
Velociraptor      2.00
Name: length_m, dtype: float64
أقدم نوع: {'species': 'Stegosaurus', 'age_million_years': 155.0}
found_in
USA         4
Mongolia    2
Canada      1
Tanzania    1
Portugal    1
Name: count, dtype: int64


## Task 5

### 1. المكرر وأسماء الأنواع
حذفت صفًا مكررًا واحدًا (F02، نفس البيانات مرتين). ووحّدت أسماء الأنواع بـ `str.strip().str.title()`، فصار `t-rex` و`T-Rex` نوعًا واحدًا (3 أحافير) بعد ما كانا يُحسبان نوعين. الشكل بعد التنظيف (9, 5).

### 2. العمر السالب
الأحفورة F05 (Stegosaurus) عمرها -150 مليون سنة. أرجّح أنه خطأ إدخال (علامة سالبة زائدة)، لأن العمر لا يكون سالبًا، والقيمة 150 قريبة من عمر Stegosaurus الآخر (155). فصحّحتها بـ `abs()` إلى 150.

### 3. العمر الناقص
ملأت العمر الناقص (F08، Velociraptor) بوسيط **نفس النوع** باستخدام `groupby(...).transform("median")`، فصار 75. وهذا أدق من وسيط كل الحفريات لأن الأنواع تعيش في عصور مختلفة، فوسيط الكل رقم لا يمثّل أي نوع. وللأمانة: Velociraptor عنده سجل واحد فقط (F04)، فالتعبئة مبنية على عينة صغيرة.

### 4. الطول
فصلت الرقم عن الوحدة بـ `str.split(" ", expand=True)`، وحوّلت القدم إلى متر (1 قدم = 0.3048 م) في عمود `length_m`: صارت `40 ft` = 12.19 م و`6.6 ft` = 2.01 م.

### 5. النتائج
- **متوسط الطول (م):** Brachiosaurus 26.00، T-Rex 12.16، Stegosaurus 9.00، Triceratops 9.00، Velociraptor 2.00.
- **أقدم نوع:** Stegosaurus (155 مليون سنة).
- **الدولة الأكثر:** USA (4 أحافير)، ثم Mongolia (2).

### 6. تفكير (Markdown)
**المشكلات المشابهة لداتا القروض:**
- **القيم الناقصة:** عمر ناقص هنا، ومثله `LoanAmount` و`Credit_History` في القروض.
- **اختلاف الكتابة:** `t-rex` و`T-Rex` تشبه قيم نصية غير موحدة، وفي القروض `3+` في `Dependents` نص وسط أرقام.
- **قيم مستحيلة:** عمر -150 مليون سنة يشبه القيم الشاذة (Outliers) في الدخل.
- **وحدات مختلطة:** متر وقدم في عمود واحد، تشبه أن نجهل وحدة `LoanAmount` (بالآلاف) فنخطئ في التحليل.
- **صف مكرر:** F02 مرتين.

**كيف أمنعها لو كنت مسؤولة عن نظام الإدخال:**
- قوائم اختيار ثابتة للأنواع والدول بدل الكتابة الحرة.
- التحقق من المدى (العمر لا يقبل سالبًا، والتقييم بين حدين).
- حقل وحدة منفصل وإلزامي، وتخزين الطول بوحدة واحدة.
- جعل الحقول المهمة إلزامية (مثل العمر والنوع).
- رقم تعريف فريد لكل سجل يمنع التكرار.